# Task B -- one versus two reinitialized encoder layers

This is a controlled ablation of the winning TAPT MuRIL recipe. The only model change is `--reinit-layers`: the control resets the top two encoder layers (the established recipe), while the variant resets only the final encoder layer.

Both runs use the same TAPT checkpoint, deduplicated Task B data, five folds, split seed 42, model seed 42, six epochs, mean+max pooling, FGM, EMA, class weighting, and `--select last`. The comparison is local five-fold OOF macro-F1; no CodaBench submission is made by this notebook.

Expected runtime: about 3--3.5 hours on a T4, including one TAPT pass. Upload this notebook to Kaggle with a GPU and Internet enabled, then use **Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, re, shutil, subprocess, sys

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    """Stream a command, optionally save its log, and fail loudly."""
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build one shared TAPT checkpoint

The TAPT checkpoint is created once and reused by both classifier runs. The defaults match the earlier `b_tapt_5f` recipe: Kannada Task B text plus OffensEval Kannada, 8 masked-LM epochs, the normal 5% perplexity split, minimum two words, and deduplication. Reusing one checkpoint ensures that the reinitialization setting is the only intended difference.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-muril"
TAPT_LOG = "artifacts/logs/reinit_tapt.log"
if pathlib.Path(TAPT_OUT).is_dir():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert pathlib.Path(TAPT_OUT).is_dir(), "TAPT checkpoint was not written"
print("TAPT checkpoint ready:", TAPT_OUT)


## 2. Train the two variants

The control is the established two-layer reset. The ablation resets only layer 12. Both runs use the fixed five-fold split and the final checkpoint rather than validation-selected checkpoints, so the reported OOF scores are the unbiased comparison.

In [ ]:
COMMON = [
    "--model", TAPT_OUT,
    "--folds", "5",
    "--seeds", "42",
    "--epochs", "6",
    "--select", "last",
    "--aux-weight", "0",
]
ARMS = [
    ("b_tapt_reinit2_5f", "2"),
    ("b_tapt_reinit1_5f", "1"),
]
for tag, n_layers in ARMS:
    log = f"artifacts/logs/{tag}.log"
    run([sys.executable, "-u", "-m", "hastika.task_b.train",
         "--tag", tag, *COMMON, "--reinit-layers", n_layers], log=log)
print("both classifier runs completed")


## 3. Validate and compare the OOF results

This cell reads each run's saved OOF probability matrix and recomputes the metric from the deduplicated training rows. It also prints per-class F1, because a small overall gain is more useful if it comes from `Violence` or `Geo-political` rather than only `Gender`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from hastika.common.preprocessing import dedupe_index

LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
train = pd.read_csv("data/raw/multiclass_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Hate Category"].tolist(), "task B")
train = train.iloc[keep].reset_index(drop=True)
y = train["Hate Category"].map(LABELS.index).to_numpy()

rows = []
for tag, n_layers in ARMS:
    run_dir = pathlib.Path("artifacts/runs") / tag
    probs = np.load(run_dir / "oof_probs.npy")
    assert probs.shape == (len(y), len(LABELS)), f"unexpected OOF shape for {tag}: {probs.shape}"
    pred = probs.argmax(1)
    report = classification_report(y, pred, labels=range(len(LABELS)),
                                  target_names=LABELS, output_dict=True, zero_division=0)
    rows.append({"run": tag, "reinit_layers": int(n_layers),
                 "macro_f1": f1_score(y, pred, average="macro"),
                 **{f"f1_{label}": report[label]["f1-score"] for label in LABELS}})

summary = pd.DataFrame(rows).set_index("run")
display(summary.round(4))
print("\nPer-class report: ")
for tag, n_layers in ARMS:
    probs = np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
    print(f"\n{tag}")
    print(classification_report(y, probs.argmax(1), labels=range(len(LABELS)),
                                target_names=LABELS, digits=3, zero_division=0))

delta = summary.loc["b_tapt_reinit1_5f", "macro_f1"] - summary.loc["b_tapt_reinit2_5f", "macro_f1"]
print(f"\nOne-layer minus two-layer macro-F1: {delta:+.4f}")
print("Treat a small single-seed difference cautiously; confirm with additional seeds before changing the recipe.")


## 4. Preserve the experiment outputs

The outputs are not a submission. Download the logs, OOF probability matrices, predictions, and this summary from Kaggle. If the one-layer model wins, record the measured values in `docs/EXPERIMENTS.md` and run a confirmation with seeds 43 and 44 before using it for a submission.

In [ ]:
# Copy only useful, reviewable outputs to Kaggle's persistent output area.
OUT = pathlib.Path("/kaggle/working/reinit_one_layer_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag, _ in ARMS:
    src = pathlib.Path("artifacts/runs") / tag
    dst = OUT / tag
    dst.mkdir(parents=True, exist_ok=True)
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        if (src / name).exists():
            shutil.copy2(src / name, dst / name)
    shutil.copy2(pathlib.Path("artifacts/logs") / f"{tag}.log", OUT / f"{tag}.log")
shutil.copy2(pathlib.Path("artifacts/logs") / "reinit_tapt.log", OUT / "reinit_tapt.log") if pathlib.Path("artifacts/logs/reinit_tapt.log").exists() else None
summary.to_csv(OUT / "reinit_comparison.csv")
print("saved outputs to", OUT)
